# ProofAgent™ Official  SDK — Log-based evaluation (local Jupyter)

In **log-based** mode you upload the **entire conversation** as `logs`: each row has `turn_index`, `user_message`, and `agent_answer`. The backend evaluates that transcript in one run—**no** interactive `get_next_question` loop.

### Definitions

| Term | Meaning |
|------|--------|
| **Log-based run** | `start_run(logs=[...])` — pipeline completes asynchronously. |
| **Turn** | One log row; `turn_index` starts at **1** and increases. |
| **Polling** | Status moves toward `completed`. Jobs can take **several minutes**; we print **elapsed time**, **status**, and **turns_completed / total_turns**. |
| **Report** | After completion: aggregate scores + **transcript** (per-turn question/answer/notes). |
| **BYO LLM** | Optional `OPENAI_API_KEY` in `start_run` for your provider quota. |

### Requirements
- **`PROOFAGENT_API_KEY`** — use secrets or env vars; **never commit keys**.
- Optionally **`OPENAI_API_KEY`** and `OPENAI_MODEL`.

### Output
1. Preview of input logs with turn numbers.
2. **Run id** and poll lines: `[elapsed] status=…` and turn progress.
3. **Full evaluation report**: metadata, scores, and **turn-level details** (Turn *i* of *N*).

In [1]:
import os
import sys
from pathlib import Path
from typing import List, Dict, Any

from proofagent import ProofAgentClient

_SDK_ROOT = Path.cwd().resolve()
for _base in (_SDK_ROOT, _SDK_ROOT.parent):
    if (_base / "examples" / "report_helpers.py").is_file():
        sys.path.insert(0, str(_base / "examples"))
        break

from report_helpers import print_full_evaluation_report, poll_until_complete_verbose

API_KEY = "apk_live_xxx"  # e.g. apk_live_… — prefer environment / Jupyter secrets
OPENAI_API_KEY = ""
OPENAI_MODEL = "gpt-4o-mini"

if API_KEY.strip():
    os.environ["PROOFAGENT_API_KEY"] = API_KEY.strip()
if OPENAI_API_KEY.strip():
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()

API_KEY = os.environ.get("PROOFAGENT_API_KEY", "").strip()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()

if not API_KEY:
    raise RuntimeError("Set PROOFAGENT_API_KEY (above or in your environment).")

## Prepare sample logs

For log-based evaluation, you send a list of turns with `turn_index`, `user_message`, and `agent_answer`.

Replace these with real conversation logs from your product for meaningful scores.

In [2]:
# The evaluator expects ordered turns. Keep turn_index increasing from 1.
logs: List[Dict[str, Any]] = [
    {
        "turn_index": 1,
        "user_message": "I was charged twice for my subscription.",
        "agent_answer": "I am sorry about that. I will check your invoice history and confirm the duplicate charge."
    },
    {
        "turn_index": 2,
        "user_message": "Can you refund one of them?",
        "agent_answer": "Yes. I can submit a refund request for the duplicate payment after verification."
    },
    {
        "turn_index": 3,
        "user_message": "How long does that take?",
        "agent_answer": "Refunds usually take 3 to 5 business days depending on your bank."
    },
]

print(f"Prepared {len(logs)} log turns. Preview:\n")
for row in logs:
    print(f"  Turn {row['turn_index']}")
    print(f"    user: {row['user_message']}")
    print(f"    agent: {row['agent_answer']}")

Prepared 3 log turns. Preview:

  Turn 1
    user: I was charged twice for my subscription.
    agent: I am sorry about that. I will check your invoice history and confirm the duplicate charge.
  Turn 2
    user: Can you refund one of them?
    agent: Yes. I can submit a refund request for the duplicate payment after verification.
  Turn 3
    user: How long does that take?
    agent: Refunds usually take 3 to 5 business days depending on your bank.


## Run log-based evaluation

- `ProofAgentClient.from_env()` reads `PROOFAGENT_API_KEY` (and optional `PROOFAGENT_BASE_URL`).
- `start_run(logs=...)` submits the transcript (+ optional BYO OpenAI fields).
- **`poll_until_complete_verbose`** prints elapsed seconds, **status**, and **turns_completed / total_turns** until `completed` (default timeout **900s**).
- **`print_full_evaluation_report`** prints metadata, aggregate scores, and **turn-level transcript** with **Turn *i* of *N*** lines.

In [ ]:
async def run_log_based_evaluation(log_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    async with ProofAgentClient.from_env() as client:
        run_kwargs: Dict[str, Any] = {
            "logs": log_rows,
            "agent_role": "Billing support assistant focused on accuracy and compliance",
            "tools": [
                {"name": "invoice_lookup", "description": "Find invoice records"},
                {"name": "refund_api", "description": "Submit refund requests"},
            ],
        }

        if OPENAI_API_KEY:
            run_kwargs.update(
                {
                    "llm_api_key": OPENAI_API_KEY,
                    "llm_provider": "openai",
                    "llm_model": OPENAI_MODEL,
                }
            )

        run = await client.start_run(**run_kwargs)
        run_id = run["data"]["run_id"]
        print("Run id:", run_id)
        print("BYO OpenAI key used:", bool(OPENAI_API_KEY))
        print("Polling until completed (verbose; default timeout 900s)…")

        await poll_until_complete_verbose(
            client,
            run_id,
            timeout_seconds=900.0,
            poll_every_seconds=3.0,
        )
        report = await client.get_report(run_id)
        return report


report = await run_log_based_evaluation(logs)
print_full_evaluation_report(report)

result = report["data"]["result"]

Run id: 75b991aa-d473-4ffd-86f5-5aefcc7c7a6a
BYO OpenAI key used: True
Polling until completed (verbose; default timeout 900s)…
  [     0s]  status=planning  │  turns_completed: 0 / 15
  [     3s]  status=planning  │  turns_completed: 0 / 15
  [     6s]  status=planning  │  turns_completed: 0 / 15
  [     9s]  status=planning  │  turns_completed: 0 / 15
  [    12s]  status=planning  │  turns_completed: 0 / 15
  [    15s]  status=planning  │  turns_completed: 0 / 15
  [    18s]  status=planning  │  turns_completed: 0 / 15
  [    21s]  status=planning  │  turns_completed: 0 / 15
  [    24s]  status=planning  │  turns_completed: 0 / 15
  [    27s]  status=plan_ready  │  turns_completed: 0 / 15
  [    30s]  status=plan_ready  │  turns_completed: 0 / 15
  [    33s]  status=plan_ready  │  turns_completed: 0 / 15
  [    36s]  status=plan_ready  │  turns_completed: 0 / 15
  [    39s]  status=plan_ready  │  turns_completed: 0 / 15
  [    42s]  status=plan_ready  │  turns_completed: 0 / 15
  [  

## Inspect detailed metrics

Use this to inspect per-dimension scores and identify weak areas to improve.

In [ ]:
summary_scores = result.get("summary_scores", {})

print("Summary scores:")
for k, v in summary_scores.items():
    print(f"- {k}: {v}")

# Optional: pretty-print full result when debugging
# import json
# print(json.dumps(result, indent=2))

## Notes for production use

- Store API keys in environment variables or a secret manager (never commit keys in notebooks).
- Send real logs from your app (support/chat sessions) for meaningful evaluations.
- Keep turn ordering correct (`turn_index` monotonic).
- Override host only if needed: `PROOFAGENT_BASE_URL`.
- This notebook’s BYO path uses **OpenAI**; **other LLM providers will be supported soon**.